# 00 - Fit Stability and Out-of-Sample Agreement

**Question.** The live config is fitted on every career, so its ratings are in-sample for all of
them. How much would a rating change if its career had *not* been in the fit, and does the fitting
procedure still produce ratings that track results when it has not seen the data?

**Method (leave-one-career-out).** For each career, fit a config on all the others
(`workshop/lib/crossval.py`) and rate the held-out career with it, so every performance gets an
out-of-sample rating alongside its in-sample one. Then:

1. how far each fold's weights, means and stds move from the live config;
2. how closely the out-of-sample ratings agree with the in-sample ones;
3. whether out-of-sample ratings are non-inferior at tracking results (rho with goal difference at
   team and player level, margin 0.05 on the lower bound of the 95% interval of the difference,
   as in `workshop/lib/gate.py`);
4. how close each derived position (wing-back, wide midfielder, CAM) is to having enough data to be
   fitted directly.

**Reading it.** Folds are small while only a few careers exist, and the interval for a different
ranking is wide, so this can show a refit is *not worse*, not that it is better. It improves as
careers and matches are added.

In [1]:
from pathlib import Path
import json
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats as stats
import seaborn as sns

project_root = Path("..").resolve().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from workshop.lib import crossval, datasets, decompose, fit, gate, inference, rating, refit, sweep, variants
from workshop.lib.report import ci, header, say

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

In [2]:
info = header()

**Validated config: `005d563f`** | run 2026-10-25

- Config fitted on: arsenal_2 1-38, ipswich_3 1-22, valencia_cf_1 1-157
- Data analysed (matches per career): arsenal_2 (38), ipswich_3 (22), valencia_cf_1 (157)

## 1. The folds: how far does dropping a career move the config?

In [3]:
GROUPS = {
    "ST": "ST", "CF": "ST", "LW": "Winger", "RW": "Winger", "LM": "WM", "RM": "WM",
    "CAM": "CAM", "CM": "CM", "CDM": "CDM", "CB": "CB", "LB": "Fullback", "RB": "Fullback",
    "LWB": "Wingback", "RWB": "Wingback", "GK": "GK",
}

LIVE_DIR = fit.LIVE_CONFIG_DIR
KEYS = ["career", "match_id", "player_id"]
N, SEED = 5000, 42
configs = crossval.loco_configs()

rows = []
for held_out, directory in configs.items():
    movement = refit.movement(LIVE_DIR, directory)
    manifest = json.loads((directory / fit.MANIFEST_FILE).read_text("utf-8"))
    fitted = next(iter(manifest["fitted_sections"].values()))["sources"]
    rows.append({
        "held out": held_out,
        "fit matches": sum(v["n_matches"] for v in fitted.values()),
        "max |weight change|": movement["max_weight_change"].max(),
        "in section": movement["max_weight_change"].idxmax(),
        "max mean shift (live stds)": movement["max_mean_shift_in_stds"].max(),
        "std ratio range": f"{movement['std_ratio_min'].min():.2f} to {movement['std_ratio_max'].max():.2f}",
    })
folds = pd.DataFrame(rows).set_index("held out")
folds

,fit matches,max |weight change|,in section,max mean shift (live stds),std ratio range
held out,,,,,
arsenal_2,179,0.041,LWB,0.486,0.00 to 1.07
ipswich_3,195,0.019,ST,0.341,0.00 to 1.10
valencia_cf_1,60,0.049,CAM,1.889,0.00 to 1.37


In [4]:
biggest = folds["max |weight change|"].idxmax()
smallest_fit = folds["fit matches"].idxmin()
same = "also" if biggest == smallest_fit else "not"
say(
    f"Dropping a career moves a single weight by at most **{folds['max |weight change|'].max():.3f}** "
    f"(held out: {biggest}, section {folds.loc[biggest, 'in section']}) and shifts a mean by up to "
    f"**{folds['max mean shift (live stds)'].max():.2f}** live stds. The fold fitted on the fewest matches "
    f"({smallest_fit}, {int(folds['fit matches'].min())}) is {same} the one that moves most."
)

Dropping a career moves a single weight by at most **0.049** (held out: valencia_cf_1, section CAM) and shifts a mean by up to **1.89** live stds. The fold fitted on the fewest matches (valencia_cf_1, 60) is also the one that moves most.

## 2. Out-of-sample vs in-sample ratings

Each performance has a rating from the live config (in-sample) and one from the fold that never saw
its career (out-of-sample). Rank agreement and size of the change, by career and by position.

In [5]:
oos = crossval.oos_ratings(configs)
inn = rating.rated_performances("pool", column="rating_in")
both = inn.merge(oos, on=KEYS, validate="one_to_one")
both = both[both["rating_in"].notna()].copy()
both["change"] = both["rating_oos"] - both["rating_in"]
both["group"] = both["primary_position"].map(GROUPS)


def agree(g: pd.DataFrame) -> pd.Series:
    return pd.Series({
        "rows": len(g),
        "rank_corr": stats.spearmanr(g["rating_in"], g["rating_oos"]).statistic,
        "mean_change": g["change"].mean(),
        "mean_abs_change": g["change"].abs().mean(),
        "share_changed": (g["change"].abs() > 0).mean(),
        "share_moved_0_5": (g["change"].abs() >= 0.5 - 1e-9).mean(),
        "max_abs_change": g["change"].abs().max(),
    })


by_career = both.groupby("career").apply(agree, include_groups=False)
by_group = both.dropna(subset=["group"]).groupby("group").apply(agree, include_groups=False)
overall = agree(both)
display(by_career)
by_group

,rows,rank_corr,mean_change,mean_abs_change,share_changed,share_moved_0_5,max_abs_change
career,,,,,,,
arsenal_2,570.000,0.993,0.050,0.074,0.553,0.000,0.400
ipswich_3,313.000,0.996,0.024,0.033,0.278,0.000,0.300
valencia_cf_1,"2,376.000",0.957,-0.159,0.214,0.820,0.106,2.500


,rows,rank_corr,mean_change,mean_abs_change,share_changed,share_moved_0_5,max_abs_change
group,,,,,,,
CAM,29.000,0.999,-0.007,0.014,0.103,0.000,0.200
CB,579.000,0.987,0.059,0.088,0.508,0.014,2.500
CDM,330.000,0.993,0.046,0.064,0.491,0.003,0.500
CM,570.000,0.969,-0.137,0.169,0.811,0.030,0.800
Fullback,568.000,0.933,-0.204,0.230,0.778,0.146,0.900
GK,217.000,0.996,-0.030,0.114,0.733,0.000,0.300
ST,333.000,0.956,-0.238,0.316,0.916,0.246,1.300
WM,72.000,0.998,0.012,0.018,0.139,0.000,0.200
Winger,561.000,0.969,-0.200,0.234,0.916,0.109,0.700


In [6]:
worst_group = by_group["mean_abs_change"].idxmax()
worst_career = by_career["mean_abs_change"].idxmax()
say(
    f"Across all {int(overall['rows'])} rated performances the out-of-sample rating agrees with the in-sample one "
    f"at rank correlation **{overall['rank_corr']:.3f}**; **{overall['share_changed']:.0%}** of ratings differ at all, "
    f"**{overall['share_moved_0_5']:.1%}** by half a point or more (largest {overall['max_abs_change']:.1f}), and the "
    f"average change is {overall['mean_change']:+.3f}. The career that depends most on its own data is "
    f"**{worst_career}** (mean absolute change {by_career.loc[worst_career, 'mean_abs_change']:.3f}); the position that "
    f"moves most is **{worst_group}** ({by_group.loc[worst_group, 'mean_abs_change']:.3f})."
)

Across all 3259 rated performances the out-of-sample rating agrees with the in-sample one at rank correlation **0.961**; **72%** of ratings differ at all, **7.7%** by half a point or more (largest 2.5), and the average change is -0.105. The career that depends most on its own data is **valencia_cf_1** (mean absolute change 0.214); the position that moves most is **ST** (0.316).

## 3. Do out-of-sample ratings still track results?

Outfield performances; criterion = goal difference. **Team level**: match-average rating, paired bootstrap
over matches. **Player level**: individual rating, cluster bootstrap over matches. The difference is
`rho(out-of-sample) - rho(in-sample)`; non-inferior when the lower bound of its 95% interval exceeds
-0.05. Per-career rows are small samples (wide intervals); the pooled row uses every match.

In [7]:
out = both[both["performance_type"] == "Outfield"].copy()
out["goal_diff"] = out["team_score"] - out["opp_score"]
out["match_key"] = out["career"] + ":" + out["match_id"].astype(str)
teams = out.groupby("match_key").agg(
    rin=("rating_in", "mean"), roos=("rating_oos", "mean"),
    gd=("goal_diff", "first"), career=("career", "first"),
)
MARGIN = gate.NON_INFERIORITY_MARGIN


def compare(t: pd.DataFrame, o: pd.DataFrame, scope: str) -> list[dict]:
    rows = []
    t_in = inference.bootstrap_spearman(t["rin"], t["gd"], N, SEED)
    t_oos = inference.bootstrap_spearman(t["roos"], t["gd"], N, SEED)
    t_d = inference.paired_delta_rho(t["rin"], t["roos"], t["gd"], N, SEED)
    p_in = inference.cluster_bootstrap_spearman(o["rating_in"], o["goal_diff"], o["match_key"], N, SEED)
    p_oos = inference.cluster_bootstrap_spearman(o["rating_oos"], o["goal_diff"], o["match_key"], N, SEED)
    p_d = inference.cluster_paired_delta_rho(o["rating_in"], o["rating_oos"], o["goal_diff"], o["match_key"], N, SEED)
    for label, a, b, d in (("team-level", t_in, t_oos, t_d), ("player-level", p_in, p_oos, p_d)):
        rows.append({
            "scope": scope, "metric": label, "matches": len(t),
            "rho in-sample": a.estimate, "rho out-of-sample": b.estimate,
            "difference": d.estimate, "ci_low": d.ci_low, "ci_high": d.ci_high,
            "non-inferior": gate.non_inferior(d.ci_low, MARGIN),
        })
    return rows


rows = compare(teams, out, "all careers pooled")
for career in sorted(teams["career"].unique()):
    rows += compare(teams[teams["career"] == career], out[out["career"] == career], career)
stability = pd.DataFrame(rows).set_index(["scope", "metric"])
stability

matches  rho in-sample  rho out-of-sample  difference  ci_low  ci_high  non-inferior
scope              metric                                                                                            
all careers pooled team-level        217          0.686              0.579      -0.108  -0.145   -0.074         False
                   player-level      217          0.242              0.219      -0.024  -0.040   -0.007          True
arsenal_2          team-level         38          0.743              0.745       0.002  -0.028    0.036          True
                   player-level       38          0.196              0.192      -0.003  -0.014    0.008          True
ipswich_3          team-level         22          0.573              0.579       0.006  -0.063    0.085         False
                   player-level       22          0.100              0.097      -0.003  -0.015    0.009          True
valencia_cf_1      team-level        157          0.728              0.697      -0.031  -0.058   -0.008         False
                   player-level      157          0.269              0.251      -0.018  -0.026   -0.009          True

In [8]:
pooled = stability.loc["all careers pooled"]
verdicts = {m: bool(pooled.loc[m, "non-inferior"]) for m in pooled.index}
widths = pooled["ci_high"] - pooled["ci_low"]
say(
    f"Pooled over {int(pooled['matches'].iloc[0])} matches: team-level rho is **{pooled.loc['team-level', 'rho in-sample']:.3f}** in-sample "
    f"and **{pooled.loc['team-level', 'rho out-of-sample']:.3f}** out-of-sample (difference "
    f"{pooled.loc['team-level', 'difference']:+.3f}, lower bound {pooled.loc['team-level', 'ci_low']:+.3f}); player-level is "
    f"**{pooled.loc['player-level', 'rho in-sample']:.3f}** and **{pooled.loc['player-level', 'rho out-of-sample']:.3f}** "
    f"(difference {pooled.loc['player-level', 'difference']:+.3f}, lower bound {pooled.loc['player-level', 'ci_low']:+.3f}). "
    f"Against the {MARGIN} margin, team-level is **{'non-inferior' if verdicts['team-level'] else 'NOT shown non-inferior'}** and "
    f"player-level is **{'non-inferior' if verdicts['player-level'] else 'NOT shown non-inferior'}**. "
    f"The pooled intervals are {widths['team-level']:.2f} (team) and {widths['player-level']:.2f} (player) wide."
)
per_career = stability.xs("team-level", level="metric").drop("all careers pooled")
within = float((per_career["difference"] * per_career["matches"]).sum() / per_career["matches"].sum())
pooled_gap = float(pooled.loc["team-level", "difference"])
level_shift = by_career["mean_change"]
say(
    f"Within each career the team-level difference averages **{within:+.3f}** (match-weighted), against the pooled "
    f"{pooled_gap:+.3f}. When the two differ, the gap comes from the folds placing careers at different rating levels "
    f"rather than from ranking matches worse within a career: mean out-of-sample minus in-sample rating is "
    + ", ".join(f"{c} {v:+.3f}" for c, v in level_shift.items())
    + ". A config fitted without a career rates it systematically higher or lower than the live config does."
)

Pooled over 217 matches: team-level rho is **0.686** in-sample and **0.579** out-of-sample (difference -0.108, lower bound -0.145); player-level is **0.242** and **0.219** (difference -0.024, lower bound -0.040). Against the 0.05 margin, team-level is **NOT shown non-inferior** and player-level is **non-inferior**. The pooled intervals are 0.07 (team) and 0.03 (player) wide.

Within each career the team-level difference averages **-0.022** (match-weighted), against the pooled -0.108. When the two differ, the gap comes from the folds placing careers at different rating levels rather than from ranking matches worse within a career: mean out-of-sample minus in-sample rating is arsenal_2 +0.050, ipswich_3 +0.024, valencia_cf_1 -0.159. A config fitted without a career rates it systematically higher or lower than the live config does.

## 4. Derived positions: how far from being fitted directly?

Wing-backs, wide midfielders and CAM are derived from a proxy position until they have enough rows of
their own; their own fit then blends in (`fit.DirectFitPolicy`).

In [9]:
manifest = json.loads((LIVE_DIR / fit.MANIFEST_FILE).read_text("utf-8"))
policy = manifest["direct_fit_policy"]
rows = []
for section, entry in manifest["derived_sections"].items():
    direct = entry.get("direct_fit", {"rows": 0, "weight": 0.0})
    rows.append({
        "section": section, "derived from": entry["proxy"], "rows of own data": direct["rows"],
        "weight on own fit": direct["weight"],
        "rows until blending starts": max(0, policy["min_rows"] - direct["rows"] + 1),
        "rows until fully own fit": max(0, policy["full_rows"] - direct["rows"]),
    })
derived = pd.DataFrame(rows).set_index("section")
derived

,derived from,rows of own data,weight on own fit,rows until blending starts,rows until fully own fit
section,,,,,
CAM,CM,29,0.000,57,221
LM,RW,69,0.000,17,181
LWB,RB,0,0.000,86,250
RM,RW,69,0.000,17,181
RWB,RB,0,0.000,86,250


In [10]:
nearest = derived["rows until blending starts"].idxmin()
say(
    f"Blending starts at {policy['min_rows']} rows and is complete at {policy['full_rows']}. "
    + "; ".join(
        f"**{s}** has {int(r['rows of own data'])} rows ("
        + ("blending" if r["weight on own fit"] > 0 else f"{int(r['rows until blending starts'])} more to start")
        + ")"
        for s, r in derived.iterrows()
    )
    + f". Closest to blending: **{nearest}**."
)

Blending starts at 85 rows and is complete at 250. **CAM** has 29 rows (57 more to start); **LM** has 69 rows (17 more to start); **LWB** has 0 rows (86 more to start); **RM** has 69 rows (17 more to start); **RWB** has 0 rows (86 more to start). Closest to blending: **LM**.

## Summary

In [11]:
passes = all(verdicts.values())
say(
    f"- Config `{info['hash'][:8]}`; folds hold out each of {', '.join(configs)} in turn.\n"
    f"- Out-of-sample ratings agree with in-sample ratings at rank correlation {overall['rank_corr']:.3f} "
    f"({overall['share_moved_0_5']:.1%} of ratings move by 0.5 or more).\n"
    f"- Non-inferiority at margin {MARGIN}: team-level **{'yes' if verdicts['team-level'] else 'not shown'}**, "
    f"player-level **{'yes' if verdicts['player-level'] else 'not shown'}** "
    f"({'the fitting procedure holds up out of sample' if passes else 'the procedure is not shown to hold up on both metrics'}).\n"
    f"- Derived positions still on proxies: {', '.join(derived.index[derived['weight on own fit'] == 0])}."
)

- Config `005d563f`; folds hold out each of arsenal_2, ipswich_3, valencia_cf_1 in turn.
- Out-of-sample ratings agree with in-sample ratings at rank correlation 0.961 (7.7% of ratings move by 0.5 or more).
- Non-inferiority at margin 0.05: team-level **not shown**, player-level **yes** (the procedure is not shown to hold up on both metrics).
- Derived positions still on proxies: CAM, LM, LWB, RM, RWB.